# D.R.O.N.A. — 06 LLM Response Quality

Compare local advising LLMs on the eval query bank:
- **Phi-3.5-mini-instruct** (primary)
- **Qwen2.5-3B-instruct** (multilingual fallback)

Metrics: RAG quality via the Ragas harness (lexical proxy without an offline
judge), plus D.R.O.N.A.'s custom **bias-mitigation** metrics. The advising path
is local-only (Ollama); cells skip gracefully if Ollama is not running.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from drona.utils.logging import setup_logging; setup_logging('WARNING')
from drona.evaluation.queries import C1_QUERIES, C2_QUERIES

eval_queries = [q.query_text for q in (C1_QUERIES[:5] + C2_QUERIES[:5])]
print(f'{len(eval_queries)} evaluation queries')

In [ ]:
# Generate responses with each model (requires Ollama + pulled models).
from drona.evaluation.bias_mitigation import evaluate_bias_mitigation
from drona.evaluation.ragas_harness import evaluate_rag

def run_model(model_name):
    from drona.advising.engine import AdvisingEngine, make_query
    eng = AdvisingEngine(model=model_name) if 'model' in AdvisingEngine.__init__.__code__.co_varnames else AdvisingEngine()
    if not eng._llm.is_available():
        raise RuntimeError('Ollama not available')
    responses, samples = [], []
    for text in eval_queries:
        resp = eng.advise(make_query(text))
        responses.append(resp)
        ctx = [c.excerpt for p in resp.pathways for c in p.citations]
        samples.append({'question': text, 'answer': resp.summary, 'contexts': ctx, 'ground_truth': None})
    return responses, samples

summary = {}
for model in ['phi3.5', 'qwen2.5:3b']:
    try:
        responses, samples = run_model(model)
        rag = evaluate_rag(samples, force_proxy=True)
        bias = evaluate_bias_mitigation(responses)
        summary[model] = {
            'faithfulness': rag.faithfulness, 'answer_relevancy': rag.answer_relevancy,
            'pathway_diversity': bias.mean_pathway_diversity,
            'hedge_freq': bias.mean_hedge_frequency,
            'refusal_rate': bias.refusal_rate, 'nepal_first_rate': bias.nepal_first_rate,
        }
        print(f'{model}: done')
    except Exception as exc:
        print(f'{model}: skipped ({exc})')

if summary:
    display(pd.DataFrame(summary).T.round(3))
else:
    print('No model ran. Start Ollama and `ollama pull phi3.5` to populate results.')

**Output:** a head-to-head table of RAG faithfulness/relevancy and bias-mitigation
behaviour for the two local models. Phi-3.5-mini is the default; Qwen2.5-3B is the
fallback for multilingual/code-switch robustness. The Ragas proxy is transparent
and dependency-free; plug an offline judge LLM into `evaluate_rag` for full Ragas.